In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
!uv add pinecone

Resolved 158 packages in 1.47s
   Building ex-0922 @ file:///D:/hanwha_0902/rag_two/ex_0922
      Built ex-0922 @ file:///D:/hanwha_0902/rag_two/ex_0922
 Downloaded pinecone
Prepared 6 packages in 813ms
Uninstalled 1 package in 44ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 6 packages in 339ms
 ~ ex-0922==0.1.0 (from file:///D:/hanwha_0902/rag_two/ex_0922)
 + h2==4.4.1
 + hpack==4.2.0
 + hyperframe==6.1.0
 + msgspec==0.21.1
 + pinecone==10.0.0


In [3]:
import os
from pinecone import Pinecone, ServerlessSpec

In [4]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "quickstart-index"

if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index= pc.Index(index_name)

In [7]:
dummy_vector = [0.01] * 1536
vectors = [
    {
        "id": "doc1",
        "values": dummy_vector,
        "metadata": {"text": "dummy vector는 더미 데이터입니다.", "category":"tech"}
    }
]
index.upsert(vectors=vectors, namespace="ex_0922")

UpsertResponse(upserted_count=1)

In [ ]:
from openai import OpenAI
import time

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

documents = [
    {"id": "doc1", "text": "Pinecone은 벡터 데이터베이스입니다."},
    {"id": "doc2", "text": "Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다."},
]

response = openai_client.embeddings.create(
    model= "text-embedding-3-small",
    input=[document["text"] for document in documents],
)

vectors = [
    {
        "id": document["id"],
        "values": embedding.embedding,
        "metadata": {"text": document["text"]},
    }
    for document, embedding in zip(documents, response.data)
]

index.upsert(vectors=vectors, namespace="example_0922")
time.sleep(2)

question = "임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?"
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

result = index.query(
    vector=question_vector,
    top_k=2,
    include_metadata=True,
)

print(f"질문: {question}\n")
for match in result.matches:
    print(f"{match.id}: {match.metadata['text']}(유사도: {match.score: .4f})")

질문: 임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?

doc2: Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다.(유사도:  0.6642)
doc1: Pinecone은 벡터 데이터베이스입니다.(유사도:  0.4800)
